# Conditional GAN for MNIST Digits

A Conditional Generative Adversarial Network (cGAN) that generates MNIST handwritten digits conditioned on class labels.

- **Generator**: Takes random noise (z_dim=100) + label embedding and produces 28x28x1 grayscale images via transposed convolutions.
- **Discriminator**: Takes an image concatenated with a label embedding on the channel axis (28x28x2) and classifies real vs fake.

Built with TensorFlow 2 / Keras.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

from keras.layers import (
    Input, Dense, Reshape, Flatten, Dropout,
    Conv2D, Conv2DTranspose, LeakyReLU, BatchNormalization,
    Embedding, Concatenate, Multiply
)
from keras.models import Model
from keras.optimizers import Adam
from keras.datasets import mnist

In [ ]:
# Hyperparameters
img_rows = 28
img_cols = 28
channels = 1
img_shape = (img_rows, img_cols, channels)
z_dim = 100
num_classes = 10

## Build the Generator

The generator takes a noise vector (z_dim=100) and a class label as input.  
The label is passed through an Embedding layer and then multiplied element-wise with the noise vector.  
The combined representation is upsampled through Dense + Conv2DTranspose layers to produce a 28x28x1 image.

In [ ]:
def build_generator(z_dim, num_classes):
    # Noise input
    noise = Input(shape=(z_dim,), name='noise_input')

    # Label input
    label = Input(shape=(1,), dtype='int32', name='label_input')

    # Label embedding: embed label into a vector of size z_dim and flatten
    label_embedding = Embedding(num_classes, z_dim, input_length=1)(label)
    label_embedding = Flatten()(label_embedding)

    # Multiply noise and label embedding element-wise
    joined = Multiply()([noise, label_embedding])

    # Foundation for 7x7 feature maps
    x = Dense(7 * 7 * 256, use_bias=False)(joined)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Reshape((7, 7, 256))(x)

    # Upsample to 14x14
    x = Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    # Upsample to 28x28
    x = Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    # Output layer: 28x28x1 with tanh activation
    x = Conv2DTranspose(1, (5, 5), strides=(1, 1), padding='same', use_bias=False, activation='tanh')(x)

    model = Model([noise, label], x, name='Generator')
    return model


generator = build_generator(z_dim, num_classes)
generator.summary()

## Build the Discriminator

The discriminator receives a 28x28x1 image and a class label.  
The label is embedded and reshaped to 28x28x1, then concatenated with the image on the channel axis, giving a 28x28x2 input.  
Conv2D layers downsample the input and a final Dense(1, sigmoid) outputs the probability of the image being real.

In [ ]:
def build_discriminator(img_shape, num_classes):
    # Image input
    img = Input(shape=img_shape, name='img_input')

    # Label input
    label = Input(shape=(1,), dtype='int32', name='label_input')

    # Label embedding: embed label and reshape to image dimensions (28x28x1)
    label_embedding = Embedding(num_classes, img_shape[0] * img_shape[1], input_length=1)(label)
    label_embedding = Flatten()(label_embedding)
    label_embedding = Reshape((img_shape[0], img_shape[1], 1))(label_embedding)

    # Concatenate image and label embedding on channel axis -> 28x28x2
    combined = Concatenate(axis=-1)([img, label_embedding])

    # Conv layers
    x = Conv2D(64, (5, 5), strides=(2, 2), padding='same')(combined)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dropout(0.3)(x)

    x = Conv2D(128, (5, 5), strides=(2, 2), padding='same')(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dropout(0.3)(x)

    x = Flatten()(x)
    x = Dense(1, activation='sigmoid')(x)

    model = Model([img, label], x, name='Discriminator')
    return model


discriminator = build_discriminator(img_shape, num_classes)
discriminator.summary()

## Compile the Models

- The discriminator is compiled with binary crossentropy and the Adam optimizer.
- For the combined (GAN) model, we freeze the discriminator weights so that only the generator is trained when we train the combined model.

In [ ]:
# Compile discriminator
discriminator.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.0002, beta_1=0.5),
    metrics=['accuracy']
)

# Freeze discriminator weights for GAN training
discriminator.trainable = False

# Build the combined model (generator -> discriminator)
noise_input = Input(shape=(z_dim,))
label_input = Input(shape=(1,), dtype='int32')

fake_img = generator([noise_input, label_input])
validity = discriminator([fake_img, label_input])

gan = Model([noise_input, label_input], validity, name='cGAN')
gan.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.0002, beta_1=0.5)
)

gan.summary()

## Load and Prepare the MNIST Dataset

In [ ]:
# Load MNIST
(X_train, y_train), (_, _) = mnist.load_data()

# Normalize to [-1, 1] (to match tanh output of generator)
X_train = (X_train.astype(np.float32) - 127.5) / 127.5
X_train = np.expand_dims(X_train, axis=-1)  # (60000, 28, 28, 1)

print('Training data shape:', X_train.shape)
print('Labels shape:', y_train.shape)

## Sample Images Function

Generates a grid of images (one per digit class 0-9) to visualize training progress.

In [ ]:
save_dir = './output/cgan_images/'
os.makedirs(save_dir, exist_ok=True)


def sample_images(generator, epoch, z_dim=100, num_classes=10):
    """Generate and display a row of sample images, one per digit class."""
    r, c = 2, 5  # 2 rows x 5 cols = 10 digits
    noise = np.random.normal(0, 1, (r * c, z_dim))
    sampled_labels = np.arange(0, num_classes).reshape(-1, 1)

    gen_imgs = generator.predict([noise, sampled_labels])

    # Rescale from [-1, 1] to [0, 1]
    gen_imgs = 0.5 * gen_imgs + 0.5

    fig, axs = plt.subplots(r, c, figsize=(10, 4))
    cnt = 0
    for i in range(r):
        for j in range(c):
            axs[i, j].imshow(gen_imgs[cnt, :, :, 0], cmap='gray')
            axs[i, j].set_title('Digit: %d' % sampled_labels[cnt])
            axs[i, j].axis('off')
            cnt += 1
    fig.savefig(os.path.join(save_dir, 'epoch_%d.png' % epoch))
    plt.show()
    plt.close()

## Training Loop

In [ ]:
def train(epochs, batch_size=128, sample_interval=1000):
    # Labels for real and fake images
    real = np.ones((batch_size, 1))
    fake = np.zeros((batch_size, 1))

    for epoch in range(epochs):
        # ---------------------
        #  Train Discriminator
        # ---------------------

        # Select a random batch of real images and their labels
        idx = np.random.randint(0, X_train.shape[0], batch_size)
        real_imgs = X_train[idx]
        labels = y_train[idx].reshape(-1, 1)

        # Generate a batch of fake images
        noise = np.random.normal(0, 1, (batch_size, z_dim))
        gen_imgs = generator.predict([noise, labels], verbose=0)

        # Train discriminator on real and fake
        d_loss_real = discriminator.train_on_batch([real_imgs, labels], real)
        d_loss_fake = discriminator.train_on_batch([gen_imgs, labels], fake)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # ---------------------
        #  Train Generator
        # ---------------------

        # Sample random noise and random labels
        noise = np.random.normal(0, 1, (batch_size, z_dim))
        sampled_labels = np.random.randint(0, num_classes, (batch_size, 1))

        # Train generator (wants discriminator to label fake images as real)
        g_loss = gan.train_on_batch([noise, sampled_labels], real)

        # Print progress
        if epoch % 200 == 0:
            print('Epoch %d [D loss: %.4f, acc.: %.2f%%] [G loss: %.4f]'
                  % (epoch, d_loss[0], 100 * d_loss[1], g_loss))

        # Save sample images at intervals
        if epoch % sample_interval == 0:
            sample_images(generator, epoch, z_dim, num_classes)

In [ ]:
# Train the Conditional GAN
train(epochs=20000, batch_size=64, sample_interval=2000)

## Save the Trained Generator

In [ ]:
generator.save('cgan_generator.h5')
print('Generator saved to cgan_generator.h5')

## Generate Final Samples

Generate a final grid of digits (0-9) using the trained generator.

In [ ]:
sample_images(generator, epoch=99999, z_dim=z_dim, num_classes=num_classes)